In [3]:
%load_ext autoreload
%autoreload 2
%matplotlib widget

import numpy as np
import matplotlib.pyplot as plt
import yaml, os, copy, time

from distgen import Generator
from matplotlib.colors import SymLogNorm

from specific_particle_tracer import SpecificParticleTracer, HemisphericalTip, FlatCathode
from specific_particle_tracer.bem.geometry import HemisphericalTipBEMGeometry, CylindricalWellBEMGeometry
from specific_particle_tracer.distributions import flat_distribution_to_hemisphere
from specific_particle_tracer.constants import ELEMENTARY_CHARGE

from specific_particle_tracer.plotting import plot_profiles, static_potential_grid, image_potential_grid

from GPT_tools.gpt_plot import gpt_plot_dist1d, gpt_plot_dist2d
from GPT_tools.image_charge import MakeMetalParticleGroup, MakeSemiconductorParticleGroup, MakeEnergyOffsetParticleGroup
from GPT_tools.image_charge import MTE_model, QE_model, getValueFromSettings, getSemiconductorEexc

template_dir = r'C:\Users\acb20\OneDrive\Desktop\Claude\BEM\distgen'
DISTGEN_INPUT_FILE = os.path.join(template_dir,'distgen.in.yaml')

# ---------------------------------------------------------------------
# Settings for multi-threading

max_workers = 8   # number of threads on your computer

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [4]:
# Some font size defaults
plt.rcParams.update({
    'font.size': 14,
    'font.family': 'sans-serif',
    'font.sans-serif': ['Arial', 'Liberation Sans', 'DejaVu Sans'],
    'mathtext.fontset': 'dejavusans',
    'axes.labelsize': 18,
    'axes.titlesize': 18,
    'xtick.labelsize': 16,
    'ytick.labelsize': 16,
    'legend.fontsize': 14,
})

## Distgen


In [38]:
settings = {}

# ---------------------------------------------------------------------
# Settings for distgen

settings['random:type'] = 'pseudo'  # 'pseudo' = not Hammersley

settings['r_dist:sigma_xy:value'] = (10 - 1.5*2)/4
settings['r_dist:sigma_xy:units'] = 'nm'
settings['r_dist:alpha:value'] = 0  # 0 = flat top, 1 = gaussian

settings['t_dist:sigma_t:value'] = 10
settings['t_dist:sigma_t:units'] = 'fs'
settings['t_dist:alpha:value'] = 0  # 0 = flat top, 1 = gaussian

# ---------------------------------------------------------------------
# Settings for cathode emission model

settings['cathode_type'] = 'semiconductor'  # metal, semiconductor, or distgen
settings['only_survivors'] = False 

# Metal only settings
if (settings['cathode_type'] == 'metal'):
    settings['photon_energy:value'] = 1.600 # 1.461
    settings['photon_energy:units'] = 'eV'
    
    settings['kT:value'] = 25e-3
    settings['kT:units'] = 'meV'
    
    settings['work_function:value'] = 1.4
    settings['work_function:units'] = 'eV'

# Semiconductor only settings
if (settings['cathode_type'] == 'semiconductor'):
    settings['photon_energy:value'] = 1.500
    settings['photon_energy:units'] = 'eV'
    
    settings['electron_affinity:value'] = -0.0
    settings['electron_affinity:units'] = 'eV'
    
    settings['energy_gap:value'] = 1.4
    settings['energy_gap:units'] = 'eV'

# distgen only settings
if (settings['cathode_type'] == 'distgen'):
    settings['start:MTE:value'] = 25
    settings['start:MTE:units'] = 'meV'

# ---------------------------------------------------------------------
# Settings for GPT

settings['gun_field:value'] = 1.0   # asymptotic field for large r
settings['gun_field:units'] = 'MV/m'

settings['plummer_radius:value'] = 1.0e-3
settings['plummer_radius:units'] = 'nm'

settings['cathode_z_offset:value'] = 3
settings['cathode_z_offset:units'] = 'nm'

settings['tmax'] = 1e-9
settings['zmax'] = 1.1e-6 

settings['R_cylinder'] = 10e-9
settings['H_cylinder'] = 10e-9

# ---------------------------------------------------------------------
# RNG settings
rng = np.random.default_rng()

In [69]:
# ---------------------------------------------------------------------
# Make initial distribution

n_particles = 10000

verbose=True

# ---------------------------------------------------------------------

settings_copy = copy.copy(settings)
settings_copy['n_particle'] = n_particles
# ---------------------------------------------------------------------

if (settings['cathode_type'] == 'metal'):
    PG_initial = MakeMetalParticleGroup(settings_copy, DISTGEN_INPUT_FILE=DISTGEN_INPUT_FILE, verbose=verbose, only_survivors=settings_copy['only_survivors'], rng=rng)
elif (settings['cathode_type'] == 'semiconductor'):
    PG_initial = MakeSemiconductorParticleGroup(settings_copy, DISTGEN_INPUT_FILE=DISTGEN_INPUT_FILE, verbose=verbose, only_survivors=settings_copy['only_survivors'], rng=rng)
elif (settings['cathode_type'] == 'distgen'):
    if (not settings_copy['only_survivors']):
        raise ValueError("ERROR: only_survivors must be used with distgen")
    PG_initial = MakeEnergyOffsetParticleGroup(settings_copy, DISTGEN_INPUT_FILE=DISTGEN_INPUT_FILE, verbose=verbose)
else:
    raise ValueError("ERROR: bad cathode type")

PG_initial.weight = ELEMENTARY_CHARGE # Just to make sure, not really needed

PG_run = copy.deepcopy(PG_initial)
PG_run.z[PG_run.r < settings_copy['R_cylinder']] = -settings_copy['H_cylinder']

#PG_run = copy.deepcopy(PG_initial)

#PG_run = flat_distribution_to_hemisphere(PG_initial, settings_copy['R_sphere'])

Adding settings["gun_field"] = 1000000.0 for use in GPT
Adding settings["cathode_z_offset"] = 3.0000000000000004e-09 for use in GPT
Adding settings["plummer_radius"] = 1.0000000000000002e-12 for use in GPT
Peak potential barrier at z = 16 nm
Eexc at surface = 0.2199970441667078, Eexc at peak = 0.134946864817795


In [70]:
geometry = CylindricalWellBEMGeometry(E_gun=-settings_copy["gun_field"], z0=settings_copy["cathode_z_offset"], R=settings_copy['R_cylinder'], H=settings_copy['H_cylinder'],
                                      field_max_length=2.0e-9,
                                      image_max_length=2.0e-9,
                                      rim_fillet_radius=5.0e-9,
                                      bottom_fillet_radius=1.5e-9
                                     )

In [ ]:
R = settings_copy['R_cylinder']
H = settings_copy['H_cylinder']

def add_minus_x(p):
    pm = copy.copy(p)
    pm[:,0] = -pm[:,0]
    p = np.append(p[::-1], pm, axis=0)
    return p

%matplotlib inline
fig, ax = plt.subplots(figsize=(10, 6))
real_profile = add_minus_x(geometry.real_profile*1e9)
image_profile = add_minus_x(geometry.image_solution.profile*1e9) if geometry.image_solution is not None else None
plot_profiles(ax, real_profile, image_profile)
ax.set_xlim(-3*R*1e9, 3*R*1e9)
ax.set_ylim(-1.8*H*1e9, 0.1*H*1e9)
ax.set_xlabel("x (nm)")
ax.set_ylabel("z (nm)")
ax.set_aspect("equal") 
plt.show()
%matplotlib widget

In [42]:
R = settings_copy['R_cylinder']
H = settings_copy['H_cylinder']
R_grid, Z_grid, V_grid = static_potential_grid(geometry.field_solver, r_max=3 * R, z_range=(-1.5*H, 2 * H))

source_pos = [0.8*R, -0.5*H]
if geometry.image_solution is not None:
    R_grid_imag, Z_grid_imag, V_grid_imag = image_potential_grid(
            geometry.image_solution,
            source_r0=source_pos[0], source_z0=source_pos[1],  # pick a point near where particles are actually emitted
            r_max=3 * R, z_range=(-1.5*H, 2 * H),
            charge=-ELEMENTARY_CHARGE, query_phi=0.0,
        )

r_bound = geometry.real_profile[:,0]
z_bound = geometry.real_profile[:,1]

In [ ]:
%matplotlib inline
fig, ax = plt.subplots()

cf = ax.contourf(
    R_grid*1e9, Z_grid*1e9, 1e3*V_grid,
    levels=20, cmap="RdBu_r"
)

cbar = fig.colorbar(
    cf,
    ax=ax,
    label="Voltage (mV)",
    shrink=0.75,   # height relative to plot
    aspect=30,     # larger = thinner
    pad=0.05       # gap between plot and colorbar
)

ax.plot(r_bound*1e9, z_bound*1e9, 'k-')

ax.set_xlabel("r (nm)")
ax.set_ylabel("z (nm)")
ax.set_aspect("equal")

ax.set_xlim(0, 3*R*1e9)
plt.show()
%matplotlib widget

In [ ]:
%matplotlib inline
if geometry.image_solution is not None:
    fig, ax = plt.subplots(figsize=(6, 6))
    cf = ax.contourf(R_grid_imag*1e9, Z_grid_imag*1e9, V_grid_imag*1e3, levels=20, cmap="RdBu_r")
    
    cbar = fig.colorbar(
        cf,
        ax=ax,
        label="Voltage (mV)",
        shrink=0.75,   # height relative to plot
        aspect=30,     # larger = thinner
        pad=0.05       # gap between plot and colorbar
    )
    
    ax.plot(source_pos[0]*1e9, source_pos[1]*1e9, marker="o", color="k", ms=7, ls="none")
    
    ax.plot(r_bound*1e9, z_bound*1e9, 'k-')

    ax.set_xlim(0, 3*R*1e9)
    ax.set_xlabel("r (nm)"); ax.set_ylabel("z (nm)"); ax.set_aspect("equal")
    plt.show()
else:
    print("geometry was built with z0=None -- no image-charge solution to plot")
%matplotlib widget

## Run

In [71]:
#t_max_plot = 1.0e-12
#t_out = np.linspace(0, t_max_plot, 1000)
t_out = None

z_screen = 1e-6

tracer = SpecificParticleTracer(
    initial_particles=PG_run,
    n_emit=1,
    geometry=geometry,
    screens=[z_screen],   # 1 micron above the tip
    t_out=t_out,
    z_max=settings_copy['zmax'],   # stop tracking a particle once it's well past the screen
    t_max=settings_copy['tmax'],
    plummer_radius=settings_copy["plummer_radius"],
    backend='gpu',
    #n_workers=max_workers,
    #rtol=1.0e-8,
    #atol=1.0e-12,
)

t_run_gpt = time.time()
screens, trajectories = tracer.run(verbose=True)
print(f'Elapsed: {time.time() - t_run_gpt} s')
print(f"{len(screens[-1])} of {len(PG_run)} particles reached the screen")

409 accepted step-batches across 10000 groups (backend=gpu)
Elapsed: 198.41050052642822 s
2429 of 10000 particles reached the screen


## Plots

In [74]:
gpt_plot_dist1d(screens[-1], 'kinetic_energy', nbins=50)

In [75]:
gpt_plot_dist2d(screens[-1], 'x', 'y', nbins=30, axis='equal')

In [ ]:
def collect_trajectories(trajectories, initial):
    """Return {id: (t_array, pos_array (n,3), mom_array (n,3))} by following each
    particle's id across the list of per-time-snapshot ParticleGroups, starting
    from its initial state in `initial`."""
    by_id = {}
    for i, pid in enumerate(initial.id):
        by_id[pid] = (
            [initial.t[i]],
            [(initial.x[i], initial.y[i], initial.z[i])],
            [(initial.px[i], initial.py[i], initial.pz[i])],
        )
    for traj in trajectories:
        for i, pid in enumerate(traj.id):
            t_list, pos_list, mom_list = by_id.setdefault(pid, ([], [], []))
            if t_list and traj.t[i] == t_list[0]:
                continue  # snapshot landed exactly on the birth time already prepended
            t_list.append(traj.t[i])
            pos_list.append((traj.x[i], traj.y[i], traj.z[i]))
            mom_list.append((traj.px[i], traj.py[i], traj.pz[i]))
    return {
        pid: (np.array(t_list), np.array(pos_list), np.array(mom_list))
        for pid, (t_list, pos_list, mom_list) in by_id.items()
    }


by_id = collect_trajectories(trajectories, PG_run)
print(f"tracked {len(by_id)} particle trajectories")

In [ ]:
r_bound = geometry.real_profile[:,0]
z_bound = geometry.real_profile[:,1]

R = settings_copy['R_cylinder']
H = settings_copy['H_cylinder']

fig, ax = plt.subplots(figsize=(6, 6))

z_plot_max = H * 2
r_plot_max = R * 2

ax.plot(r_bound * 1e9, z_bound * 1e9, color="0.6", lw=2)

for pid, (t, pos, mom) in by_id.items():
    ax.plot(np.sqrt(pos[:, 0]**2 + pos[:, 1]**2) * 1e9, pos[:, 2] * 1e9, lw=0.8, alpha=0.7)

ax.set_xlabel("r [nm]")
ax.set_ylabel("z [nm]")
ax.set_xlim(0, r_plot_max*1e9)
ax.set_ylim(-H*1e9, z_plot_max*1e9)
ax.set_aspect("equal")
plt.show()